# Import and predefining

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

set_config(display="text")

random_state = 42

In [ ]:
@dataclass
class ModelResult:
    name: str
    classification_report: pd.DataFrame
    confusion_matrix: np.ndarray
    accuracy_score: float

In [ ]:
def plot_heatmap(
        dataframe: pd.DataFrame,
        figsize: tuple[int, int],
        cmap: str | None = None,
        vmin: float | None = None,
        vmax: float | None = None,
        rotation: int = 45,
        xticklabels: list[str] | bool = True,
        yticklabels: list[str] | bool = True,
        title_name: str | None = None
    ) -> None:
    plt.figure(figsize=figsize)

    sns.heatmap(
        dataframe, 
        cmap=cmap, 
        vmin=vmin, 
        vmax=vmax, 
        linewidths=0.5,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7},
        xticklabels=xticklabels,
        yticklabels=yticklabels
    )
    plt.xticks(rotation=rotation, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    if title_name:
        plt.title(title_name)
    plt.show()

def plot_table(
        cellText: np.ndarray | pd.DataFrame,
        rowLabels: list | None = None,
        colLabels: list | None = None,
        title_name: str | None = None,
        fontsize: int = 10,
        scale: tuple[float, float] = (1.7, 1.5),
        figsize: tuple[int, int] = (8, 3)
    ) -> None:
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')

    table = ax.table(
        cellText=cellText,
        rowLabels=rowLabels,
        colLabels=colLabels,
        cellLoc='center',
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(*scale)
    if title_name:
        plt.title(title_name)
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

label_encoder_previous_lab = LabelEncoder()
scaler = StandardScaler()
previous_lab_path = Path("../lab1/data/Date_Fruit_Datasets.xlsx")
raw_data_previous_lab = pd.read_excel(previous_lab_path)

target_name = "Class"
features_previous_lab = raw_data_previous_lab.drop(target_name, axis=1)
target_previous_lab = raw_data_previous_lab[target_name]

target_encoded_previous_lab = label_encoder_previous_lab.fit_transform(target_previous_lab)

X_train_previous_lab, X_test_previous_lab, y_train_previous_lab, y_test_previous_lab  = train_test_split(features_previous_lab, target_encoded_previous_lab, random_state=random_state, test_size=0.25, stratify=target_encoded_previous_lab)

X_train_previous_lab_scaled = scaler.fit_transform(X_train_previous_lab)
X_test_previous_lab_scaled = scaler.transform(X_test_previous_lab)


# 1. Зниження розмірності та візуалізація даних
1. Візьміть датасет, з яким ви працювали в лабораторній роботі №1.
1. Застосуйте `sklearn.decomposition.PCA` для зниження розмірності даних.
1. На отриманих даних навчіть один з алгоритмів класифікації з Lab 1 та порівняйте результати класифікації "до" та "після" PCA.
1. Зробіть висновки: як PCA вплинув на якість класифікації;  наскільки вдалося зменшити кількість ознак;  чи змінився час навчання моделі.
1. Для того самого датасету застосуйте `sklearn.manifold.TSNE` для зниження розмірності до 2D.
1. Візуалізуйте отримані дані, використовуючи кольори для позначення класів, та поясніть результат.

## Візьміть датасет, з яким ви працювали в лабораторній роботі №1

## Застосуйте `sklearn.decomposition.PCA` для зниження розмірності даних

In [ ]:
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.svm import SVC

svc_lab1 = SVC(C=10, gamma = 0.01, kernel = 'rbf')
svc_lab1.fit(X_train_previous_lab_scaled, y_train_previous_lab)

pca = PCA(
    random_state=random_state
)
X_train_previous_lab_scaled_decomposed = pca.fit_transform(X_train_previous_lab_scaled)

svc_pca: SVC = clone(svc_lab1)
svc_pca.fit(X_train_previous_lab_scaled_decomposed, y_train_previous_lab)

## На отриманих даних навчіть один з алгоритмів класифікації з Lab 1 та порівняйте результати класифікації "до" та "після" PCA

In [ ]:
from typing import Any


def compare_results(models: dict[str, Any]):
    for model_name, model in models.items():
        y_pred = model.predict(X_test_previous_lab_scaled)

        model_result = ModelResult(
            name=model_name,
            classification_report=pd.DataFrame(classification_report(y_test_previous_lab, y_pred, output_dict=True)),
            confusion_matrix=confusion_matrix(y_test_previous_lab, y_pred),
            accuracy_score=accuracy_score(y_test_previous_lab, y_pred) # pyright: ignore[reportArgumentType]
        )
        model_result.classification_report.columns=[*label_encoder_previous_lab.classes_, "accuracy", "macro avg", "weighted avg"]
        
        plot_table(
            cellText=model_result.classification_report.round(2),
            title_name=f"Classification Report {model_result.name}"
        )

        plot_heatmap(
            dataframe=model_result.confusion_matrix, # pyright: ignore[reportArgumentType]
            figsize=(4, 4),
            cmap="coolwarm",
            rotation=45,
            xticklabels=label_encoder_previous_lab.classes_, # pyright: ignore[reportArgumentType]
            yticklabels=label_encoder_previous_lab.classes_, # pyright: ignore[reportArgumentType]
            title_name=f"Confusion Matrix {model_result.name}"
        )

        print(f"Accuracy Score {model_result.name}: {model_result.accuracy_score}")


In [ ]:
models_pca = {"Before": svc_lab1, "After": svc_pca}
compare_results(models_pca) 

## Зробіть висновки: як PCA вплинув на якість класифікації;  наскільки вдалося зменшити кількість ознак;  чи змінився час навчання моделі.

## Для того самого датасету застосуйте `sklearn.manifold.TSNE` для зниження розмірності до 2D

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(
    random_state=random_state
)

X_train_previous_lab_scaled_decomposed = tsne.fit_transform(X_train_previous_lab_scaled)

svc_tsne: SVC = clone(svc_lab1)
svc_tsne.fit(X_train_previous_lab_scaled_decomposed, y_train_previous_lab)

## Візуалізуйте отримані дані, використовуючи кольори для позначення класів, та поясніть результат

In [ ]:
models_tsne = {"Before": svc_lab1, "After": svc_tsne}
compare_results(models_tsne) 

# 2. Кластерний аналіз
1. Візьміть датасет з лабораторної роботи №1.
1. Не використовуйте мітки класів (target) під час кластеризації. Виконайте кластеризацію даних за допомогою алгоритму K-Means.
1. Визначте доцільну кількість кластерів за допомогою Elbow Method та/або Silhouette Score.
1. Візуалізуйте отримані кластери у 2D за допомогою PCA або t-SNE.
1. Після виконання кластеризації порівняйте отримані кластери зі справжніми класами датасету.
1. Зробіть висновок: чи відповідає знайдена кластерна структура реальним класам?

Якщо датасет містить велику кількість класів, для спрощення аналізу можна залишити 2–3 класи, обґрунтувавши свій вибір.
Важливо: кластер і клас — не одне й те саме. Не очікується, що K-Means обов'язково відтворить початкові класи датасету. Потрібно проаналізувати та пояснити отриманий результат.

## Візьміть датасет з лабораторної роботи №1

## Не використовуйте мітки класів (target) під час кластеризації. Виконайте кластеризацію даних за допомогою алгоритму K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Визначте доцільну кількість кластерів за допомогою Elbow Method та/або Silhouette Score

In [ ]:
Error = []
Score = []
for i in range(1, 12):
    kmeans = KMeans(n_clusters = i, random_state=random_state)
    kmeans_labels = kmeans.fit_predict(features_previous_lab)

    silhouette_avg = silhouette_score(features_previous_lab, kmeans_labels)

    Error.append(kmeans.inertia_)
    Score.append(silhouette_avg)

plt.plot(range(1, 12), Error, marker = 'o')
plt.xlabel('N of clusters')
plt.ylabel('Inertia')
plt.grid()
plt.show()

plt.plot(range(1, 12), Score, marker = 'o')
plt.xlabel('N of clusters')
plt.ylabel('Silhouette average')
plt.grid()
plt.show()

## Візуалізуйте отримані кластери у 2D за допомогою PCA або t-SNE

## Після виконання кластеризації порівняйте отримані кластери зі справжніми класами датасету

## Зробіть висновок: чи відповідає знайдена кластерна структура реальним класам?

# 3. Обробка та класифікація текстових даних
1. Завантажте набір текстових даних з мітками класів. Проведіть передобробку текстів (наприклад, видалення стоп-слів та пунктуації).
2. За допомогою WordCloud візуалізуйте найбільш характерні слова/словосполучення для кожного класу та поясніть отримані результати.
3. Векторизуйте тексти, наприклад за допомогою `sklearn.feature_extraction.text.TfidfVectorizer`.
4. Навчіть модель класифікації текстів та оцініть якість класифікації за відповідними метриками.

## Завантажте набір текстових даних з мітками класів. Проведіть передобробку текстів (наприклад, видалення стоп-слів та пунктуації)

In [ ]:
path = Path("data/Date_Fruit_Datasets.xlsx")
raw_data = pd.read_excel(path)

## За допомогою WordCloud візуалізуйте найбільш характерні слова/словосполучення для кожного класу та поясніть отримані результати

In [ ]:
from wordcloud import WordCloud


def visualize(df: pd.DataFrame, label: str) -> None:
    text = ''
    for msg in df[df['target'] == label]['text']:
        msg = msg.lower()
        text += msg + ' '
    wordcloud = WordCloud(width=600, height=400, background_color="white").generate(text)
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.show()

visualize('')


## Векторизуйте тексти, наприклад за допомогою `sklearn.feature_extraction.text.TfidfVectorizer`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_df=500, min_df=10)

tfidf_train = tfidf_vectorizer.fit_transform(newsgroups_train.data)
tfidf_test =  tfidf_vectorizer.transform(newsgroups_test.data)

## Навчіть модель класифікації текстів та оцініть якість класифікації за відповідними метриками

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression().fit(tfidf_train, newsgroups_train.target)

predicted = clf.predict(tfidf_test)
print(classification_report(newsgroups_test.target, predicted))